# Setup Environment

In [1]:
!pip install pandas geopandas pyiceberg[s3fs,pyarrow] shapely

In [2]:
import pandas as pd
import geopandas as gpd
import shapely.wkb
import os
from pyiceberg.catalog.rest import RestCatalog

ICEBERG_REST_URI = os.getenv("ICEBERG_REST_URI", "http://iceberg-rest:8181")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin")
ICEBERG_NS = "mch"


def _catalog():
    """
    Return a PyIceberg REST catalog connected to the local Iceberg REST server.

    PyIceberg catalog properties mirror the REST catalog spec:
      uri       – REST catalog endpoint
      s3.*      – S3FileIO properties for reading/writing the actual data files
    """

    return RestCatalog(
        name="local",
        **{
            "uri": ICEBERG_REST_URI,
            "s3.endpoint": MINIO_ENDPOINT,
            "s3.access-key-id": AWS_KEY,
            "s3.secret-access-key": AWS_SECRET,
            "s3.path-style-access": "true",
            # Tell PyArrow's S3FileSystem to use the MinIO endpoint too
            "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO",
        },
    )

cat = _catalog()

In [3]:
cat.list_tables(ICEBERG_NS)

[('mch', 'point-forecast_jp2000d0'),
 ('mch', 'point-forecast_jp2000d0_cleaned'),
 ('mch', 'point-forecast_meta_point'),
 ('mch', 'point-forecast_meta_point_cleaned'),
 ('mch', 'point-forecast_point_denormalised')]

In [4]:
TABLE = f"{ICEBERG_NS}.point-forecast_jp2000d0"
table = cat.load_table(TABLE)


# PyIceberg → PyArrow → pandas
df = table.scan().to_pandas()
print(f"Table schema:\n{df.dtypes}\nRow count: {len(df)}")
display(df.head(20))

Table schema:
point_id                       int64
point_type_id                  int64
Date                             str
jp2000d0                       int64
_ingested_at     datetime64[us, UTC]
_source_file                     str
dtype: object
Row count: 50625


,point_id,point_type_id,Date,jp2000d0,_ingested_at,_source_file
0,1,1,202605060000,33,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
1,1,1,202605070000,3,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
2,1,1,202605080000,3,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
3,1,1,202605090000,3,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
4,1,1,202605100000,3,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
5,1,1,202605110000,9,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
6,1,1,202605120000,9,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
7,1,1,202605130000,9,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
8,1,1,202605140000,10,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...
9,10,1,202605060000,20,2026-07-23 12:51:11.848956+00:00,s3://raw-data/mch/point-forecast/20260506/vnut...


In [5]:
TABLE = f"{ICEBERG_NS}.point-forecast_jp2000d0_cleaned"
table = cat.load_table(TABLE)

# PyIceberg → PyArrow → pandas
df = table.scan().to_pandas()
print(f"Table schema:\n{df.dtypes}\nRow count: {len(df)}")
display(df.head(20))

Table schema:
point_id                  int64
point_type_id             int64
jp2000d0                  int64
date             datetime64[us]
dtype: object
Row count: 50625


,point_id,point_type_id,jp2000d0,date
0,1,1,33,2026-05-06
1,1,1,3,2026-05-07
2,1,1,3,2026-05-08
3,1,1,3,2026-05-09
4,1,1,3,2026-05-10
5,1,1,9,2026-05-11
6,1,1,9,2026-05-12
7,1,1,9,2026-05-13
8,1,1,10,2026-05-14
9,10,1,20,2026-05-06


In [6]:
TABLE = f"{ICEBERG_NS}.point-forecast_meta_point"
table = cat.load_table(TABLE)

# PyIceberg → PyArrow → pandas
df = table.scan().to_pandas()
display(df.head(20))

,point_id,point_type_id,station_abbr,postal_code,point_name,point_type_de,point_type_fr,point_type_it,point_type_en,point_height_masl,point_coordinates_lv95_east,point_coordinates_lv95_north,point_coordinates_wgs84_lat,point_coordinates_wgs84_lon,_ingested_at,_source_file
0,1,1,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,46.792661,9.679014,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
1,2,1,RAG,NaN,Bad Ragaz,Station,Station,Stazione,Station,497.0,2756911.0,1209351.0,47.016631,9.502594,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
2,3,1,HAI,NaN,Salen-Reutenen,Station,Station,Stazione,Station,719.0,2719100.0,1279047.0,47.651242,9.023911,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
3,4,1,HLL,NaN,Hallau,Station,Station,Stazione,Station,419.0,2677457.0,1283472.0,47.697278,8.470464,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
4,6,1,DEM,NaN,Delémont,Station,Station,Stazione,Station,439.0,2593270.0,1244543.0,47.351706,7.349567,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
5,7,1,EBK,NaN,Ebnat-Kappel,Station,Station,Stazione,Station,623.0,2726348.0,1237176.0,47.273389,9.108494,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
6,8,1,ELM,NaN,Elm,Station,Station,Stazione,Station,958.0,2732266.0,1198424.0,46.923742,9.175347,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
7,9,1,EIN,NaN,Einsiedeln,Station,Station,Stazione,Station,911.0,2699984.0,1221068.0,47.133042,8.756556,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
8,10,1,ANT,NaN,Andermatt,Station,Station,Stazione,Station,1435.0,2687445.0,1165044.0,46.630914,8.580553,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
9,11,1,MER,NaN,Meiringen,Station,Station,Stazione,Station,589.0,2655844.0,1175930.0,46.732222,8.169247,2026-06-16 10:59:44.930483+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...


In [7]:
TABLE = f"{ICEBERG_NS}.point-forecast_meta_parameters"
table = cat.load_table(TABLE)

# PyIceberg → PyArrow → pandas
df = table.scan().to_pandas()
display(df.head(20))

,parameter_shortname,parameter_description_de,parameter_description_fr,parameter_description_it,parameter_description_en,parameter_group_de,parameter_group_fr,parameter_group_it,parameter_group_en,parameter_granularity,parameter_decimals,parameter_datatype,parameter_unit,_ingested_at,_source_file
0,dkl010h0,Windrichtung; Stundenmittel,Direction du vent; moyenne horaire,Direzione del vento; media oraria,Wind direction; hourly mean,Wind,Vent,Vento,Wind,H,0,Integer,°,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
1,fu3010h0,Windgeschwindigkeit skalar; Stundenmittel in km/h,Vitesse du vent scalaire; moyenne horaire en km/h,Velocità del vento scalare; media oraria in km/h,Wind speed scalar; hourly mean in km/h,Wind,Vent,Vento,Wind,H,1,Float,km/h,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
2,fu3010h1,Böenspitze (Sekundenböe); Stundenmaximum in km/h,Rafale (intégration 1 s); maximum horaire en km/h,Raffica del vento (su un secondo); massima ora...,Gust peak (one second); hourly maximum in km/h,Wind,Vent,Vento,Wind,H,1,Float,km/h,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
3,fu3q10h0,"Windgeschwindigkeit; Stundenmittel, 10% Quanti...","Vitesse du vent; moyenne horaire, quantil 10% ...","Velocità del vento; media oraria, quantile 10%...","Wind speed; hourly mean, 10% quantile in km/h",Wind,Vent,Vento,Wind,H,1,Float,km/h,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
4,fu3q10h1,"Böenspitze; Stundenmaximum, 10% Quantil in km/h","Rafale; maximum horaire, quantil 10% en km/h","Raffica del vento; massima oraria, quantile 10...","Gust peak; hourly maximum, 10% quantile in km/h",Wind,Vent,Vento,Wind,H,1,Float,km/h,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
5,fu3q90h0,"Windgeschwindigkeit; Stundenmittel, 90% Quanti...","Vitesse du vent; moyenne horaire, quantil 90% ...","Velocità del vento; media oraria, quantile 90%...","Wind speed; hourly mean, 90% quantile in km/h",Wind,Vent,Vento,Wind,H,1,Float,km/h,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
6,fu3q90h1,"Böenspitze; Stundenmaximum, 90% Quantil in km/h","Rafale; maximum horaire, quantil 90% en km/h","Raffica del vento; massima oraria, quantile 90...","Gust peak; hourly maximum, 90% quantile in km/h",Wind,Vent,Vento,Wind,H,1,Float,km/h,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
7,gre000h0,Globalstrahlung; Stundenmittel,Rayonnement global; moyenne horaire,Radiazione globale; media oraria,Global radiation; hourly mean,Strahlung,Rayonnement,Radiazione,Radiation,H,0,Integer,W/m²,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
8,jp2000d0,MeteoSchweiz-Piktogramm-Nummer; Tageswert (gül...,"Numéro du pictogramme MétéoSuisse, valeur jour...","Numero del pittogramma di MeteoSvizzera, valor...","MeteoSwiss pictogram number, daily value (vali...",Grafik,Graphique,Grafica,Graphics,D,0,Integer,Code,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...
9,jww003i0,MeteoSchweiz-Piktogramm-Nummer; über 3 Stunden,"Numéro du pictogramme MétéoSuisse, valeur sur ...","Numero del pittogramma di MeteoSvizzera, valor...","MeteoSwiss-Icon, weathertype, preceding 3 hour...",Grafik,Graphique,Grafica,Graphics,H,0,Integer,Code,2026-05-11 11:53:50.027953+00:00,s3://raw-data/mch/point-forecast/ogd-local-for...


In [7]:
# 1. Load the Table
TABLE = f"{ICEBERG_NS}.point-forecast_meta_point_cleaned"
table = cat.load_table(TABLE)

# PyIceberg → PyArrow → pandas
df = table.scan().to_pandas()

# 3. Convert to GeoPandas
# Assuming the geometry column is named 'geom' and is in WKB format
df['geometry'] = df['geometry'].apply(lambda x: shapely.wkb.loads(x))
gdf = gpd.GeoDataFrame(df, geometry='geometry')

# Set CRS if necessary
gdf.crs = "EPSG:2056"


display(gdf.head(20))


,point_id,point_type_id,station_abbr,postal_code,point_name,point_type_de,point_type_fr,point_type_it,point_type_en,point_height_masl,point_coordinates_lv95_east,point_coordinates_lv95_north,geometry
0,1,1,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
1,2,1,RAG,NaN,Bad Ragaz,Station,Station,Stazione,Station,497.0,2756911.0,1209351.0,POINT (2756911 1209351)
2,3,1,HAI,NaN,Salen-Reutenen,Station,Station,Stazione,Station,719.0,2719100.0,1279047.0,POINT (2719100 1279047)
3,4,1,HLL,NaN,Hallau,Station,Station,Stazione,Station,419.0,2677457.0,1283472.0,POINT (2677457 1283472)
4,6,1,DEM,NaN,Delémont,Station,Station,Stazione,Station,439.0,2593270.0,1244543.0,POINT (2593270 1244543)
5,7,1,EBK,NaN,Ebnat-Kappel,Station,Station,Stazione,Station,623.0,2726348.0,1237176.0,POINT (2726348 1237176)
6,8,1,ELM,NaN,Elm,Station,Station,Stazione,Station,958.0,2732266.0,1198424.0,POINT (2732266 1198424)
7,9,1,EIN,NaN,Einsiedeln,Station,Station,Stazione,Station,911.0,2699984.0,1221068.0,POINT (2699984 1221068)
8,10,1,ANT,NaN,Andermatt,Station,Station,Stazione,Station,1435.0,2687445.0,1165044.0,POINT (2687445 1165044)
9,11,1,MER,NaN,Meiringen,Station,Station,Stazione,Station,589.0,2655844.0,1175930.0,POINT (2655844 1175930)


In [25]:
TABLE = f"{ICEBERG_NS}.point-forecast_point_denormalised"
table = cat.load_table(TABLE)

# PyIceberg → PyArrow → pandas
df = table.scan().to_pandas()


# 3. Convert to GeoPandas
# Assuming the geometry column is named 'geom' and is in WKB format
df['geometry'] = df['geometry'].apply(lambda x: shapely.wkb.loads(x))
gdf = gpd.GeoDataFrame(df, geometry='geometry')

# Set CRS if necessary
gdf.crs = "EPSG:2056"


display(gdf.head(20))

print(f"Table schema:\n{gdf.dtypes}\nRow count: {len(df)}")
display(df.head(20))

,point_id,point_type_id,jp2000d0,date,station_abbr,postal_code,point_name,point_type_de,point_type_fr,point_type_it,point_type_en,point_height_masl,point_coordinates_lv95_east,point_coordinates_lv95_north,geometry
0,1,1,33,2026-05-06,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
1,1,1,3,2026-05-07,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
2,1,1,3,2026-05-08,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
3,1,1,3,2026-05-09,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
4,1,1,3,2026-05-10,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
5,1,1,9,2026-05-11,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
6,1,1,9,2026-05-12,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
7,1,1,9,2026-05-13,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
8,1,1,10,2026-05-14,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
9,10,1,20,2026-05-06,ANT,NaN,Andermatt,Station,Station,Stazione,Station,1435.0,2687445.0,1165044.0,POINT (2687445 1165044)


Table schema:
point_id                                 int64
point_type_id                            int64
jp2000d0                                 int64
date                            datetime64[us]
station_abbr                               str
postal_code                            float64
point_name                                 str
point_type_de                              str
point_type_fr                              str
point_type_it                              str
point_type_en                              str
point_height_masl                      float64
point_coordinates_lv95_east            float64
point_coordinates_lv95_north           float64
geometry                              geometry
dtype: object
Row count: 50661


,point_id,point_type_id,jp2000d0,date,station_abbr,postal_code,point_name,point_type_de,point_type_fr,point_type_it,point_type_en,point_height_masl,point_coordinates_lv95_east,point_coordinates_lv95_north,geometry
0,1,1,33,2026-05-06,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
1,1,1,3,2026-05-07,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
2,1,1,3,2026-05-08,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
3,1,1,3,2026-05-09,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
4,1,1,3,2026-05-10,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
5,1,1,9,2026-05-11,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
6,1,1,9,2026-05-12,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
7,1,1,9,2026-05-13,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
8,1,1,10,2026-05-14,ARO,NaN,Arosa,Station,Station,Stazione,Station,1878.0,2771031.0,1184830.0,POINT (2771031 1184830)
9,10,1,20,2026-05-06,ANT,NaN,Andermatt,Station,Station,Stazione,Station,1435.0,2687445.0,1165044.0,POINT (2687445 1165044)
